# Agricultural Analytics — Google Colab Runner

Run the existing **FastAPI backend** and **Streamlit dashboard** from one Colab notebook. The project files remain regular Python modules, so you can keep editing them outside Colab.

> The project uses FastAPI memory instead of a database. All live history resets when the Colab runtime disconnects.

## 1. Upload and extract the project

When prompted, upload `drought-monitoring-system.zip`. If the project is already extracted under `/content`, this cell reuses it.

In [ ]:
from pathlib import Path
import zipfile

from google.colab import files

CONTENT_ROOT = Path('/content')
PROJECT_ROOT = CONTENT_ROOT / 'drought-monitoring-system'

if not (PROJECT_ROOT / 'backend' / 'app' / 'main.py').exists():
    print('Upload drought-monitoring-system.zip')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise FileNotFoundError('No ZIP file was uploaded.')

    zip_path = CONTENT_ROOT / zip_names[0]
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(CONTENT_ROOT)

if not (PROJECT_ROOT / 'backend' / 'app' / 'main.py').exists():
    candidates = list(CONTENT_ROOT.glob('**/backend/app/main.py'))
    if not candidates:
        raise FileNotFoundError('Could not locate backend/app/main.py in the uploaded ZIP.')
    PROJECT_ROOT = candidates[0].parents[2]

print(f'Project ready: {PROJECT_ROOT}')

## 2. Install the project dependencies

This installs the existing backend and dashboard requirements. `pyngrok` is added only to create optional public URLs.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(PROJECT_ROOT / 'backend' / 'requirements.txt'),
    '-r', str(PROJECT_ROOT / 'dashboard' / 'requirements.txt'),
    'pyngrok>=7,<8',
])
print('Dependencies installed.')

## 3. Load credentials safely

In Colab, open the key icon named **Secrets**, add the values you use, and grant notebook access. Do not paste secret keys into notebook cells.

Add `GEMINI_API_KEY` for forecast explanations and chat. Open-Meteo needs no key, and no database secrets are required.

In [ ]:
import os
from google.colab import userdata

SECRET_NAMES = [
    'FARM_LATITUDE',
    'FARM_LONGITUDE',
    'SERIAL_DEVICE_ID',
    'SOIL_SENSOR_DRY_RAW',
    'SOIL_SENSOR_WET_RAW',
    'WATER_SENSOR_EMPTY_RAW',
    'WATER_SENSOR_FULL_RAW',
    'GEMINI_API_KEY',
    'LINE_CHANNEL_ACCESS_TOKEN',
    'LINE_CHANNEL_SECRET',
    'NGROK_AUTH_TOKEN',
]

def read_colab_secret(name):
    try:
        return userdata.get(name) or ''
    except Exception:
        return ''

loaded = []
for secret_name in SECRET_NAMES:
    secret_value = read_colab_secret(secret_name)
    if secret_value:
        os.environ[secret_name] = secret_value
        loaded.append(secret_name)

os.environ['STORAGE_BACKEND'] = 'memory'
os.environ['BACKEND_URL'] = 'http://127.0.0.1:8000'
print('Loaded:', ', '.join(loaded) if loaded else 'no optional secrets')
print('Secret values are hidden.')

## 4. Start FastAPI and Streamlit

Both applications run as background processes in the same Colab runtime. The dashboard talks to FastAPI at `http://127.0.0.1:8000`.

In [ ]:
import time
import httpx

def stop_process(process):
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()

stop_process(globals().get('backend_process'))
stop_process(globals().get('dashboard_process'))

backend_log = open('/content/fastapi.log', 'w')
dashboard_log = open('/content/streamlit.log', 'w')
runtime_env = os.environ.copy()

backend_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=str(PROJECT_ROOT / 'backend'),
    env=runtime_env,
    stdout=backend_log,
    stderr=subprocess.STDOUT,
)

dashboard_process = subprocess.Popen(
    [
        sys.executable, '-m', 'streamlit', 'run', 'app.py',
        '--server.address', '0.0.0.0',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--browser.gatherUsageStats', 'false',
    ],
    cwd=str(PROJECT_ROOT / 'dashboard'),
    env=runtime_env,
    stdout=dashboard_log,
    stderr=subprocess.STDOUT,
)

for attempt in range(30):
    try:
        response = httpx.get('http://127.0.0.1:8000/health', timeout=2, trust_env=False)
        response.raise_for_status()
        break
    except httpx.HTTPError:
        time.sleep(1)
else:
    raise RuntimeError('FastAPI did not start. Run the log cell below for details.')

print('FastAPI is healthy on port 8000.')
print('Streamlit is starting on port 8501.')

## 5. Create public links (optional)

Add `NGROK_AUTH_TOKEN` to Colab Secrets before running this cell. The Streamlit URL opens the dashboard. The FastAPI URL exposes `/docs` and can later receive external sensor or LINE webhook traffic.

In [ ]:
from pyngrok import ngrok

ngrok_token = os.getenv('NGROK_AUTH_TOKEN', '')
if not ngrok_token:
    raise ValueError('Add NGROK_AUTH_TOKEN to Colab Secrets, then rerun the secrets cell.')

ngrok.kill()
ngrok.set_auth_token(ngrok_token)
api_tunnel = ngrok.connect(8000, bind_tls=True)
dashboard_tunnel = ngrok.connect(8501, bind_tls=True)

FASTAPI_PUBLIC_URL = api_tunnel.public_url
STREAMLIT_PUBLIC_URL = dashboard_tunnel.public_url

print(f'Dashboard: {STREAMLIT_PUBLIC_URL}')
print(f'FastAPI docs: {FASTAPI_PUBLIC_URL}/docs')

## 6. Test the backend

The health response should show `storage: memory`. This test sends one temporary sensor reading without a database.

In [ ]:
health = httpx.get('http://127.0.0.1:8000/health', timeout=5, trust_env=False)
print('Health:', health.status_code, health.json())

simulated_reading = {
    'device_id': 'COLAB_SIMULATOR_01',
    'soil_moisture': 18,
    'water_level': 4,
}
sensor_response = httpx.post(
    'http://127.0.0.1:8000/api/readings',
    json=simulated_reading,
    timeout=15,
    trust_env=False,
)
print('Sensor request:', sensor_response.status_code, sensor_response.text)

## 7. View logs or stop the services

Run the first cell if an app fails to start. Run the second cell when you are finished.

In [ ]:
print('--- FastAPI log ---')
print(Path('/content/fastapi.log').read_text(errors='replace')[-4000:])
print('--- Streamlit log ---')
print(Path('/content/streamlit.log').read_text(errors='replace')[-4000:])

In [ ]:
stop_process(globals().get('dashboard_process'))
stop_process(globals().get('backend_process'))
try:
    ngrok.kill()
except NameError:
    pass
print('FastAPI, Streamlit, and ngrok have been stopped.')